# Part 3: Exploratory Data Analysis

In this section, we will visually explore the data to uncover trends or patterns behind flight delays between 2020 and 2025.

Ouputs were cleared to allow for upload to GitHub.

In [ ]:
# Import necessary libraries
from typing import List
import datetime as dt
import pandas as pd
import pandasql as ps
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

SEED = 100

In [ ]:
# Split from total notebook -> cell inserted to get starting data
final_cleaned_df = pd.read_parquet("../data/processed/cleaned_eda.parquet")

In [ ]:
# Copy cleaned df for visualization
viz_df = final_cleaned_df.copy()

### 3.1 Target Variable Distribution

First, we examine the distribution of our target variable to understand the balance of classes in our dataset.

This bar chart displays the class distribution of the binary target variable for the flight delay prediction task. Flights are classified into two categories: those that arrived within 15 minutes of their scheduled time (IsDelayed = 0) and those that arrived more than 15 minutes late (IsDelayed = 1).


In [ ]:
# Target variable distribution
plt.figure(figsize = (12,6))
ax = sns.countplot(data = viz_df, x = "IsDelayed")

# Add labels to bars
for p in ax.patches:
  height = p.get_height()
  ax.text(
      p.get_x() + p.get_width() / 2,
      height,
      f"{height} min",
      ha = "center",
      va = "bottom"
  )

plt.xticks([0,1], ["Arrived Within 15 Minutes", "Arrived Later than 15 Minutes"])
plt.xlabel("")
plt.ylabel("Count")
plt.title("Distribution of Binary Target (Delay >15m)")

plt.show()


The dataset exhibits significant class imbalance, with 384,215 flights (approximately 80.8%) arriving within the 15-minute threshold and 91,159 flights (approximately 19.2%) experiencing delays exceeding 15 minutes. This roughly 4:1 ratio indicates that the majority of flights at Philadelphia International Airport during 2020-2025 arrived on time or with minimal delays.

### 3.2 Correlation Matrix for Numeric Variables

Next, we examine the correlation matrix to identify relationships between numeric features and the target variable, as well as potential multicollinearity among predictors.

This heatmap displays Pearson correlation coefficients between all numeric variables in the dataset. The color intensity indicates the strength and direction of linear relationships, with red representing positive correlations, blue representing negative correlations, and white indicating no correlation. Each cell shows the correlation value ranging from -1 (perfect negative correlation) to +1 (perfect positive correlation).

In [ ]:
corrs = viz_df.corr(numeric_only=True)

plt.figure(figsize = (12,6))

sns.heatmap(
    corrs,
    annot = True,
    fmt = ".2f",
    cmap = "coolwarm",
    vmin = -1,
    vmax = 1,
    linewidth = 0.50,
    annot_kws={"fontsize": 8}
)

plt.title("Correlation Matrix for Numeric Variables")
plt.show()

In [ ]:
viz_df.corr(numeric_only = True)["IsDelayed"].sort_values().to_frame()

The correlation analysis reveals several important insights for delay prediction. As expected, ArrDelay (the continuous delay variable) shows the strongest correlation with IsDelayed (0.53), which validates our binary classification threshold. Among weather-related features, origin airport conditions show stronger associations with delays than destination conditions, with origin_precipitation_hours (0.09), origin_weather_code (0.09), and origin_precipitation_sum (0.07) demonstrating the most notable positive correlations. Interestingly, Year shows a modest positive correlation (0.11) with delays, potentially reflecting changing patterns during the 2020-2025 period. The matrix also highlights important multicollinearity concerns. Several weather variables are highly correlated with each other, such as dest_precipitation_sum and dest_rain_sum (0.84), and dest_wind_gusts_10m_max and dest_wind_speed_10m_max (0.91). Similarly, flight characteristics like CRSElapsedTime, AirTime, and Distance show strong intercorrelations (0.95+), which is expected since these variables measure related aspects of flight duration. These multicollinearities may require feature selection or dimensionality reduction techniques to avoid redundancy in the final model.


### 3.3 Correlation With Target
To better visualize feature importance, we isolate the correlations between all numeric features and the target variable IsDelayed.

This heatmap displays the Pearson correlation coefficients between each numeric feature and the binary target variable IsDelayed, sorted from most negative to most positive correlation. The color gradient emphasizes the strength and direction of each relationship, with red indicating positive correlations and blue indicating negative or weak correlations.

In [ ]:
sns.heatmap(
    viz_df.corr(numeric_only = True)["IsDelayed"].sort_values().to_frame(),
    annot = True,
    fmt = ".2f",
    cmap = "coolwarm"
)

plt.xticks(fontsize=8)
plt.yticks(fontsize=6)

plt.title("Correlation with Target", fontsize=12)
plt.tight_layout()
plt.show()

The visualization confirms that most features exhibit weak to moderate correlations with flight delays. The strongest predictor is naturally ArrDelay (0.53), the continuous arrival delay variable from which our binary target was derived. Among actionable predictive features, Year shows the highest correlation (0.11), followed by weather-related variables at the origin airport: origin_precipitation_hours (0.09), origin_weather_code (0.09), and origin_precipitation_sum (0.07). This pattern suggests that adverse weather conditions at departure locations have a more significant impact on delays than conditions at destination airports.

### 3.4 Distribution of Numerical Features

We now examine the distributions of individual numeric features using an interactive histogram with a dropdown menu to explore each variable.

This interactive visualization allows exploration of the distribution shape for each of the 28 numeric variables in the dataset (excluding geographic coordinates and categorical features). Users can select different features from the dropdown menu to view their respective histograms, which display the frequency distribution of values across the entire dataset.


In [ ]:
cols = [c for c in viz_df.columns if c not in ["origin_latitude", "origin_longitude"] and viz_df[c].dtype != "object"]  # your 29 numeric vars

fig = go.Figure()

# Hidden traces
for i, col in enumerate(cols):
    fig.add_trace(go.Histogram(
        x = viz_df[col],
        name = col,
        visible = (i == 0)  # show only the first by default
    ))

# Dropdown
buttons = []
for i, col in enumerate(cols):
    buttons.append(dict(
        method = "update",
        label = col,
        args = [
            {"visible": [j == i for j in range(len(cols))]},  # toggle traces
            {"title": f"Distribution of {col}"}]
        ))

fig.update_layout(
    updatemenus = [dict(
        active = 0,
        buttons = buttons,
        x = 1.25,
        xanchor = "right",
        y = 1.2,
        yanchor = "top"
    )],

    title = f"Distribution of {cols[0]}",
    xaxis_title = "Value",
    yaxis_title = "Count"
)

fig.show()


### 3.5 Geographic Distribution of Flight Delays by Origin Airport

We examine the spatial patterns of flight delays across origin airports to identify geographic hotspots and regional trends in delay rates.

This interactive geographic scatter plot visualizes delay patterns for all origin airports in the dataset. Each point represents an airport positioned by its geographic coordinates, with bubble size indicating the percentage of delayed flights (delay rate) and color intensity representing the average arrival delay in minutes. The color scale ranges from lighter shades (lower delays) to darker red (higher delays), while larger bubbles indicate airports with higher proportions of delayed departures.

In [ ]:
# Look at delay rate by airport and then graph
airport_delay_rates = viz_df.groupby("Origin").agg(
    PercentDelayed = ("IsDelayed", "mean"),
    AvgArrDelay = ("ArrDelay", "mean"),
    Latitude = ("origin_latitude", "first"),
    Longitude = ("origin_longitude", "first")).reset_index()

fig = px.scatter_geo(
    airport_delay_rates,
    lat = "Latitude",
    lon = "Longitude",
    color = "AvgArrDelay",
    size = "PercentDelayed",
    color_continuous_scale = px.colors.sequential.OrRd ,
    range_color = [min(airport_delay_rates["AvgArrDelay"]), max(airport_delay_rates["AvgArrDelay"])],
    hover_name = "Origin",
    hover_data = {
        "PercentDelayed": True,
        "AvgArrDelay": True,
        "Latitude": False,
        "Longitude": False
    },
    projection = "albers usa",
    title = "Which airports are most likely to have delayed flights?",
    size_max = 15
)

fig.update_coloraxes(
    colorbar = dict(
        title = dict(
            text = "Average Arrival Delay (min)\n",
            side = "top"
        ),
        tickformat = ".2f",
        lenmode = "pixels",
        thickness = 20,
        x = 0.95,
        y = 0.50
    )
)

fig.update_traces(
    hovertemplate=(
        "<b>Airport: %{hovertext}</b><br>" +
        "Delay Rate: %{customdata[0]:.1%}<br>" +
        "Average Arrival Delay: %{customdata[1]:.1f} min<br>" +
        "<extra></extra>"
    )
)

fig.update_geos(
    visible=True,
    resolution=50,
    showcountries=True,
    showland=True,
    landcolor="lightgray",
    lakecolor="white"
)


fig.show()

### 3.6 Airline Performance Comparison: Delay Rates and Average Delays

We compare airline performance to identify which carriers exhibit the highest and lowest delay rates and average arrival delays.

This dual bar chart visualization displays two key performance metrics for each airline in the dataset. The left panel shows the proportion of flights delayed more than 15 minutes (delay rate), while the right panel displays the average arrival delay in minutes. Airlines are sorted by delay rate from lowest to highest, with color coding to highlight the best performer (green), worst performer (red), and middle performers (orange) for each metric.

In [ ]:
airline_delay_rates = viz_df.groupby("airline_name").agg(
    PropDelayed = ("IsDelayed", "mean"),
    AvgArrDelay = ("ArrDelay", "mean")
).reset_index()

In [ ]:
airline_delay_rates_sorted = airline_delay_rates.sort_values(by = "PropDelayed", ascending = True)

# Function to assign colors
def assign_color(value, col):
    if value == airline_delay_rates_sorted[col].min():
        return "best"
    elif value == airline_delay_rates_sorted[col].max():
        return "worst"
    else:
        return "middle"

airline_delay_rates_sorted["hue_prop"] = (
    airline_delay_rates_sorted["PropDelayed"]
    .apply(lambda x: assign_color(x, "PropDelayed")))

airline_delay_rates_sorted["hue_avg"] = (
    airline_delay_rates_sorted["AvgArrDelay"]
    .apply(lambda x: assign_color(x, "AvgArrDelay")))

# Color mapping
color_dict = {"best": "#22bb33", "middle": "#f0ad4e", "worst": "#bb2124"}

# Plot side by side figures
fig, axes = plt.subplots(ncols = 2, figsize = (16,6), sharey = True)

sns.barplot(
    data = airline_delay_rates_sorted,
    x = "PropDelayed",
    y = "airline_name",
    hue = "hue_prop",
    palette = color_dict,
    dodge = False,
    legend = False,
    ax = axes[0]
)

axes[0].set_xlabel("Proportion of flights delayed >15 min", fontsize = 12)
axes[0].set_title("Proportion of Flights Delayed by Airline", fontsize = 14)

# Add value labels
for i, value in enumerate(airline_delay_rates_sorted["PropDelayed"]):
    axes[0].text(value + 0.005, i, f"{value:.1%}", va='center', fontsize = 10)

sns.barplot(
    data = airline_delay_rates_sorted,
    x = "AvgArrDelay",
    y = "airline_name",
    hue = "hue_avg",
    palette = color_dict,
    dodge = False,
    legend = False,
    ax = axes[1]
)
axes[1].set_xlabel("Average arrival delay (min)", fontsize = 12)
axes[1].set_title("Average Arrival Delay by Airline", fontsize = 14)

# Add value labels
for i, value in enumerate(airline_delay_rates_sorted["AvgArrDelay"]):
  axes[1].text(value + 0.2, i, f"{value:.1f}", va='center', fontsize = 10)


for ax in axes:
  ax.set_ylabel("")
  sns.despine(ax = ax, left = True, bottom = True)
  ax.grid(axis = "x", linestyle = "--", alpha = 0.50)

plt.tight_layout()
plt.show()

The analysis reveals substantial variation in airline performance across carriers. Endeavor Air Inc. demonstrates the best delay performance with only 9.2% of flights delayed and an average arrival delay of -7.7 minutes, indicating that this carrier typically arrives ahead of schedule. In stark contrast, Frontier Airlines Inc. exhibits the worst performance with 30.4% of flights delayed and an average arrival delay of 20.7 minutes, more than three times the delay rate of the best performer. Interestingly, the rankings for delay rate and average delay magnitude do not perfectly align, suggesting different operational characteristics among airlines. For example, several carriers like Republic Airline, Mesa Airlines Inc., and Expressjet Airlines LLC show relatively low delay rates (11-16%) but have negative average delays, indicating they often arrive early when not delayed. Conversely, airlines like American Airlines Inc. and Spirit Air Lines show moderate delay rates (19-23%) but higher average delays when delays do occur (9.6 and 8.6 minutes respectively).Mid-tier carriers cluster around a 15-20% delay rate with average delays between 0-6 minutes, representing typical industry performance during the 2020-2025 period. This distribution suggests that airline identity is a meaningful predictor for the delay model, as operational practices, fleet characteristics, route networks, and scheduling philosophies vary significantly across carriers. Including airline as a categorical feature will allow the model to capture these carrier-specific delay patterns.

### 3.7 Delay Patterns by Origin Airport and Day of Week
We explore the interaction between origin airports and day of week to identify temporal patterns in delay rates across different locations.

This density heatmap visualizes the proportion of delayed flights (sum of IsDelayed) for each combination of origin airport (y-axis) and day of week (x-axis, where 0 = Monday through 6 = Sunday). The color intensity represents the delay rate, with darker purple indicating lower delay rates (closer to 0% delayed) and brighter yellow-green indicating higher delay rates (closer to 100% delayed). Each cell shows the average delay probability for flights departing from a specific airport on a specific day.


In [ ]:
df_pivot = viz_df.groupby(['Origin','DayOfWeek'])['IsDelayed'].mean().reset_index()
px.density_heatmap(df_pivot, x='DayOfWeek', y='Origin', z='IsDelayed', color_continuous_scale='Viridis')

 Extreme values like the CHA-Tuesday combination (100% delay rate) likely represent cells with very small sample sizes, perhaps only one or two flights operated on that route, day combination, and all happened to be delayed. This highlights an important consideration for modeling: not all airport-day combinations have sufficient data volume to provide reliable delay rate estimates.

### 3.8 Average Delay Rate by Origin Airport (Ranked)

We examine the distribution of delay rates across all origin airports by ranking them from highest to lowest average delay.

In [ ]:
airport_stats = viz_df.groupby('Origin').agg(
    AvgDelay=('IsDelayed', 'mean'),
    NumFlights=('FlightDate', 'count'),
    Latitude=('origin_latitude','first'),
    Longitude=('origin_longitude','first')
).reset_index()
airport_stats = airport_stats.sort_values('AvgDelay', ascending=False)
px.line(airport_stats, x='Origin', y='AvgDelay', title='Avg Delay by Airport (sorted)')


This line plot displays the average delay rate (proportion of flights delayed more than 15 minutes) for each origin airport, sorted in descending order from highest to lowest delay rates. The x-axis shows individual airport codes, while the y-axis represents the delay rate ranging from 0 (no delays) to approximately 0.6 (60% of flights delayed). The steep initial decline followed by a gradual flattening illustrates the distribution of delay performance across the airport network. The plot reveals a highly skewed distribution of delay rates across airports, with a few problematic airports exhibiting dramatically higher delay rates than the majority. The leftmost airports (CHA, BOI, VPS) show delay rates between 50-60%, representing significant operational challenges at these locations. However, the curve drops sharply within the first 10-15 airports, falling to around 25-30% delay rates, before gradually declining more slowly across the remaining airports.


### 3.9 Monthly Delay Patterns Across Years (Animated)

We explore temporal trends in delay rates by examining monthly patterns across different years in an animated visualization.

This animated line plot displays the percentage of delayed flights for each month, with animation frames allowing users to observe how monthly delay patterns evolve across different years (2020-2025). The x-axis represents months (1=January through 12=December), the y-axis shows the proportion of flights delayed, and the animation frame selector allows switching between years. The y-axis range is fixed across all frames to enable direct comparison of delay magnitudes between years.

In [ ]:
month_year_df = viz_df.groupby(["Year", "Month"]).agg(
    PctDelayed = ("IsDelayed", "mean")
).reset_index()

fig = px.line(
    month_year_df,
    x="Month",
    y="PctDelayed",
    animation_frame="Year",
    markers=True,
    range_y=[0, month_year_df["PctDelayed"].max() + 0.02]   # keep Y axis consistent across frames
)

fig.show()

The animation reveals dramatic variation in both seasonal patterns and overall delay rates across the 2020-2025 period, reflecting the evolution of air travel operations during and after the COVID-19 pandemic.

**Year 2020**: The delay pattern shows relatively low rates early in the year (around 13-16% in January-February), followed by a sharp drop in April-May (3-5%), likely corresponding to the initial pandemic period with drastically reduced flight volumes. Delay rates gradually increase through the summer and fall, ending the year around 11% in December. This U-shaped pattern reflects the unprecedented disruption and subsequent recovery of the aviation industry.

**Year 2021**: The data shows a dramatic summer peak with delay rates climbing from around 10% in early months to 23-25% in June-August, the highest sustained delays observed in the dataset. This likely reflects the surge in pent-up travel demand combined with operational challenges as airlines ramped up service following pandemic disruptions. Fall months show improvement (15% in September-October) before rising again toward year-end.

**Year 2022**: The pattern exhibits more stability with delay rates consistently in the 20-24% range throughout most of the year, showing a broad summer peak from May through August. This suggests the industry was still grappling with elevated delays as travel demand remained high while operational challenges persisted.

**Year 2023**: This year displays the most pronounced seasonal pattern, with a sharp summer spike reaching 31% in July—the highest single-month delay rate in the entire dataset. The dramatic rise from around 17% in February to the July peak, followed by a sharp decline to 14% by October, indicates strong seasonal effects. The elevated summer delays may reflect record travel volumes, weather disruptions, or operational capacity constraints.

**Year 2024**: The data shows a distinctly different pattern with summer delays (June-July reaching 32-33%) remaining elevated but followed by a notable improvement in fall, with delay rates dropping to 15% by October-November before a slight uptick in December (22%). This suggests potential improvements in operational resilience during the latter part of the year.

**Year 2025**: Based on partial data (January-July only), the year shows a concerning upward trend with delay rates climbing from 19% in January to 34% by July, representing the highest delay rate observed for any July in the dataset. This steep monotonic increase suggests either worsening operational conditions or data quality issues that warrant investigation.

### 3.10 Delay Rate for Top 20 Busiest Airports
We focus on the 20 airports with the highest flight volumes to examine delay rates among major hubs in the network.

This bar chart displays the delay rates (proportion of flights delayed more than 15 minutes) for the 20 busiest origin airports by flight count, sorted in descending order by delay rate. By focusing on high-volume airports, this analysis provides insights into delay patterns at the major hubs that handle the bulk of air traffic and have the greatest impact on overall network performance.

In [ ]:
top_airports = viz_df['Origin'].value_counts().head(20).index
df_top = viz_df[viz_df['Origin'].isin(top_airports)]
px.bar(df_top.groupby('Origin')['IsDelayed'].mean().reset_index().sort_values(by = "IsDelayed", ascending = False),
       x='Origin', y='IsDelayed', title='Delay Rate for Top 20 Airports')



Among the busiest airports, delay rates range from approximately 25% at the worst performers down to around 14% at the best, demonstrating that even high-volume airports exhibit substantial variation in operational efficiency. The top three worst-performing major airports—SJU (San Juan), MCO (Orlando), and DEN (Denver)—all experience delay rates around 25%, which is notably higher than the dataset average of approximately 19%. These airports likely face unique operational challenges: SJU may be affected by Caribbean weather patterns and international operations complexity, MCO handles massive leisure travel volumes that can strain capacity, and DEN is known for weather-related challenges including snow and high-altitude conditions.

### 3.11 Airport Flight Volume vs Delay Rate
We investigate the relationship between airport traffic volume and delay rates to understand whether busier airports experience systematically different delay patterns.


This scatter plot displays the relationship between flight volume (x-axis) and delay rate (y-axis) for all origin airports in the dataset. Each bubble represents an individual airport, with bubble size proportional to the number of flights operated from that airport. The x-axis shows the total flight count, while the y-axis represents the proportion of flights delayed more than 15 minutes.

In [ ]:
# Group by Origin, aggregate count and average delay flag
df_airport = viz_df.groupby('Origin').agg(NumFlights = ('FlightDate','count'),PctDelayed = ('IsDelayed','mean')).reset_index()
px.scatter(df_airport, x='NumFlights', y='PctDelayed', size='NumFlights', title='Airport Flight Volume vs Delay Rate')

The visualization reveals no clear linear relationship between flight volume and delay rates, challenging the intuitive assumption that busier airports necessarily experience more delays. The scatter pattern shows substantial variation in delay rates across all volume levels, with delay rates ranging from approximately 10% to 30% regardless of whether an airport handles hundreds or tens of thousands of flights.

### 3.12 Distribution of Arrival Delays by Delay Classification

We examine the distribution of continuous arrival delay values, separated by binary delay classification, to understand the relationship between the continuous and categorical representations of delays.

This overlaid histogram displays the distribution of ArrDelay (arrival delay in minutes) with color-coding to distinguish between flights classified as on-time (IsDelayed = 0, blue) and delayed (IsDelayed = 1, red). The x-axis shows arrival delay in minutes, where negative values indicate early arrivals and positive values indicate late arrivals. The y-axis represents the count of flights, and the overlay mode allows visualization of how the two classes overlap in the delay distribution.

In [ ]:
px.histogram(viz_df, x='ArrDelay', color='IsDelayed', nbins=50, barmode='overlay')


The histogram reveals a heavily right-skewed distribution concentrated near zero, which is typical for arrival delay data. The vast majority of flights (over 300,000, shown by the tall blue bar) cluster around zero delay, indicating that most flights arrive close to their scheduled time. The binary classification threshold at 15 minutes creates a clear separation: the blue (on-time) distribution dominates the region from approximately -50 to +15 minutes, while the red (delayed) distribution appears for values above 15 minutes.

### 3.13 Flight Volume Distribution by Day of Week

We examine how total flight volumes (both delayed and on-time) are distributed across different days of the week to understand weekly traffic patterns.


This bar chart displays the total number of flights operated on each day of the week, with the x-axis showing days from Monday through Sunday and the y-axis representing the count of flights. The relatively uniform bar heights indicate how flight schedules are distributed throughout the week. Note that despite the title mentioning "Arrival Delay," this visualization actually shows total flight counts per day rather than specifically delayed flights.

In [ ]:
day_map = {0: "Monday", 1: "Tuesday", 2: "Wednesday", 3: "Thursday",
           4: "Friday", 5: "Saturday", 6: "Sunday"}
grouped_df = viz_df.groupby("DayOfWeek")["IsDelayed"].size().reset_index()
grouped_df["DoW"] = grouped_df["DayOfWeek"].map(day_map)

fig = px.bar(grouped_df,
       x="DoW", y='IsDelayed', title='Arrival Delay by Airport and Day of Week',
       labels = {
           "DoW": "Day of the Week",
           "IsDelayed": "Number of Delayed Flights"
       })
fig.show()


The distribution reveals relatively consistent flight volumes across the week, with most days ranging between 63,000 and 72,000 flights. Monday, Thursday, Friday, and Sunday show the highest volumes at approximately 70,000-72,000 flights, representing peak travel days. These patterns align with typical business and leisure travel behaviors: Mondays and Fridays capture business travelers starting and ending their work weeks, while Thursdays and Sundays reflect return travel from extended business trips and weekend getaways.